In [1]:
import gc
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset

nltk.download('punkt')
try:
    nltk.download('punkt_tab')
except Exception:
    pass

# По договоренности считаем именно на нужной видеокарте.
device = torch.device("cuda:0")
print('device:', device)

PROJECT_DIR = Path('.') if Path('data').exists() else Path('Текущий контроль 9-10')
DATA_DIR = PROJECT_DIR / 'data'
MODELS_DIR = PROJECT_DIR / 'saved_models'
MODELS_DIR.mkdir(exist_ok=True)

criterion = nn.CrossEntropyLoss()


def preprocess_text(text: str):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z.!?]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return [token for token in word_tokenize(text) if not re.fullmatch(r'[.!?]', token)]


def clear_memory(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def evaluate_classifier(model, loader, criterion, device):
    model.eval()
    losses = []
    y_true = []
    y_pred = []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            losses.append(loss.item() * X_batch.size(0))
            preds = logits.argmax(dim=1)
            y_true.extend(y_batch.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())
    return sum(losses) / len(loader.dataset), accuracy_score(y_true, y_pred)


def train_classifier(model, train_loader, test_loader, criterion, optimizer, epochs, device, patience=None, min_delta=0.0):
    model = model.to(device)
    history = {'train_loss': [], 'test_loss': [], 'test_acc': []}
    best_test_loss = float('inf')
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()
        train_loss_sum = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * X_batch.size(0)

        train_loss = train_loss_sum / len(train_loader.dataset)
        test_loss, test_acc = evaluate_classifier(model, test_loader, criterion, device)
        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        print(f'epoch {epoch + 1}/{epochs}: train_loss={train_loss:.4f} test_loss={test_loss:.4f} test_acc={test_acc:.4f}')

        if test_loss < best_test_loss - min_delta:
            best_test_loss = test_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if patience is not None and epochs_without_improvement >= patience:
                print(f'early stopping at epoch {epoch + 1}')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return history


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], label='train')
    axes[0].plot(history['test_loss'], label='test')
    axes[0].set_title(title + ' loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(history['test_acc'], label='test_acc', color='green')
    axes[1].set_title(title + ' accuracy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def show_text_predictions(model, texts, labels, vectorize_fn, label_decoder, device, n=5, title='Examples'):
    print(title)
    model.eval()
    for text, label in list(zip(texts, labels))[:n]:
        with torch.no_grad():
            vector = vectorize_fn(text).unsqueeze(0).to(device)
            probs = torch.softmax(model(vector), dim=1).squeeze(0)
            pred_idx = int(torch.argmax(probs).item())
        true_label = label_decoder(label)
        pred_label = label_decoder(pred_idx)
        preview = str(text).replace('\n', ' ')[:180]
        print(f'true={true_label} | pred={pred_label} | text={preview}...')


def make_class_weights(targets):
    counts = pd.Series(targets).value_counts().sort_index()
    weights = 1.0 / torch.tensor(counts.values, dtype=torch.float32)
    weights = weights / weights.sum() * len(weights)
    return weights


SyntaxError: unterminated string literal (detected at line 137) (3576889926.py, line 137)

## 1. Представление и предобработка текстовых данных в виде последовательностей

1.1 Представьте первое предложение из строки `text` как последовательность из индексов слов, входящих в это предложение

In [ ]:
text = 'Select your preferences and run the install command. Stable represents the most currently tested and supported version of PyTorch. Note that LibTorch is only available for C++'

all_tokens = sorted(set(preprocess_text(text)))
token_to_idx = {token: idx for idx, token in enumerate(all_tokens)}
first_sentence = sent_tokenize(text)[0]
first_sentence_tokens = preprocess_text(first_sentence)
first_sentence_indices = [token_to_idx[token] for token in first_sentence_tokens]

print('first_sentence_tokens:', first_sentence_tokens)
print('first_sentence_indices:', first_sentence_indices)


1.2 Представьте первое предложение из строки `text` как последовательность векторов, соответствующих индексам слов. Для представления индекса в виде вектора используйте унитарное кодирование. В результате должен получиться двумерный тензор размера `количество слов в предложении` x `количество уникальных слов`

In [ ]:
text = 'Select your preferences and run the install command. Stable represents the most currently tested and supported version of PyTorch. Note that LibTorch is only available for C++'

all_tokens = sorted(set(preprocess_text(text)))
token_to_idx = {token: idx for idx, token in enumerate(all_tokens)}
first_sentence = sent_tokenize(text)[0]
first_sentence_tokens = preprocess_text(first_sentence)
first_sentence_indices = [token_to_idx[token] for token in first_sentence_tokens]

sentence_one_hot = torch.zeros(len(first_sentence_indices), len(all_tokens), dtype=torch.float32)
for row, idx in enumerate(first_sentence_indices):
    sentence_one_hot[row, idx] = 1.0

print('sentence_one_hot shape:', sentence_one_hot.shape)
print(sentence_one_hot)


1.3 Решите задачу 1.2, используя модуль `nn.Embedding`

In [ ]:
embedding = nn.Embedding(num_embeddings=len(all_tokens), embedding_dim=4)
embedded_sentence = embedding(torch.tensor(first_sentence_indices, dtype=torch.long))
print('embedded_sentence shape:', embedded_sentence.shape)
print(embedded_sentence)


## 2. Классификация фамилий по национальности (ConvNet)

Датасет: https://disk.yandex.ru/d/owHew8hzPc7X9Q?w=1

2.1 Считать файл `surnames/surnames.csv`. 

2.2 Закодировать национальности числами, начиная с 0.

2.3 Разбить датасет на обучающую и тестовую выборку

2.4 Реализовать класс `Vocab` (токен = __символ__)
  * добавьте в словарь специальный токен `<PAD>` с индексом 0
  * при создании словаря сохраните длину самой длинной последовательности из набора данных в виде атрибута `max_seq_len`

2.5 Реализовать класс `SurnamesDataset`
  * метод `__getitem__` возвращает пару: <последовательность индексов токенов (см. 1.1 ), номер класса> 
  * длина каждой такой последовательности должна быть одинаковой и равной `vocab.max_seq_len`. Чтобы добиться этого, дополните последовательность справа индексом токена `<PAD>` до нужной длины

2.6. Обучить классификатор.
  
  * Для преобразования последовательности индексов в последовательность векторов используйте `nn.Embedding`. Рассмотрите два варианта: 
    - когда токен представляется в виде унитарного вектора и модуль `nn.Embedding` не обучается
    - когда токен представляется в виде вектора небольшой размерности (меньше, чем размер словаря) и модуль `nn.Embedding` обучается

  * Используйте одномерные свертки и пулинг (`nn.Conv1d`, `nn.MaxPool1d`)
    - обратите внимание, что `nn.Conv1d` ожидает на вход трехмерный тензор размерности `(batch, embedding_dim, seq_len)`

2.7 Измерить точность на тестовой выборке. Проверить работоспособность модели: прогнать несколько фамилий студентов группы через модели и проверить результат. Для каждой фамилии выводить 3 наиболее вероятных предсказания.

In [ ]:
class Vocab:
    def __init__(self, tokens, pad_token='<PAD>', unk_token=None, min_freq=1):
        counts = Counter(tokens)
        specials = [pad_token]
        if unk_token is not None:
            specials.append(unk_token)
        vocab_tokens = [token for token, count in counts.items() if count >= min_freq and token not in specials]
        self.idx_to_token = specials + sorted(vocab_tokens)
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)}
        self.vocab_len = len(self.idx_to_token)
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.max_seq_len = None


In [ ]:
class SurnamesDataset(Dataset):
    def __init__(self, X, y, vocab: Vocab):
        self.X = list(X)
        self.y = list(y)
        self.vocab = vocab

    def vectorize(self, surname):
        pad_idx = self.vocab.token_to_idx[self.vocab.pad_token]
        unk_idx = self.vocab.token_to_idx.get(self.vocab.unk_token, pad_idx)
        indices = [self.vocab.token_to_idx.get(char, unk_idx) for char in surname.lower()[:self.vocab.max_seq_len]]
        if len(indices) < self.vocab.max_seq_len:
            indices += [pad_idx] * (self.vocab.max_seq_len - len(indices))
        return torch.tensor(indices, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.vectorize(self.X[idx]), torch.tensor(self.y[idx], dtype=torch.long)


surnames_candidates = [
    DATA_DIR / 'surnames.csv',
    DATA_DIR / 'surnames' / 'surnames.csv',
]

surnames_path = next((path for path in surnames_candidates if path.exists()), None)
if surnames_path is None:
    print('Не найден файл surnames.csv. Ожидаемые пути:', surnames_candidates)
else:
    surnames_df = pd.read_csv(surnames_path)
    label_encoder = LabelEncoder()
    surnames_df['target'] = label_encoder.fit_transform(surnames_df['nationality'])

    X_train, X_test, y_train, y_test = train_test_split(
        surnames_df['surname'],
        surnames_df['target'],
        test_size=0.2,
        random_state=42,
        stratify=surnames_df['target'],
    )

    char_vocab = Vocab(''.join(X_train.astype(str).str.lower().tolist()), pad_token='<PAD>', unk_token='<UNK>')
    char_vocab.max_seq_len = int(X_train.astype(str).str.len().max())

    train_dataset = SurnamesDataset(X_train.tolist(), y_train.tolist(), char_vocab)
    test_dataset = SurnamesDataset(X_test.tolist(), y_test.tolist(), char_vocab)

    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

    class SurnameConvClassifier(nn.Module):
        def __init__(self, embedding_layer, embedding_dim, num_classes):
            super().__init__()
            self.embedding = embedding_layer
            self.conv1 = nn.Conv1d(embedding_dim, 128, kernel_size=3, padding=1)
            self.conv2 = nn.Conv1d(128, 64, kernel_size=3, padding=1)
            self.pool = nn.AdaptiveMaxPool1d(1)
            self.dropout = nn.Dropout(0.3)
            self.fc = nn.Linear(64, num_classes)

        def forward(self, x):
            x = self.embedding(x)
            x = x.transpose(1, 2)
            x = torch.relu(self.conv1(x))
            x = torch.relu(self.conv2(x))
            x = self.pool(x).squeeze(-1)
            x = self.dropout(x)
            return self.fc(x)

    num_classes = len(label_encoder.classes_)
    class_weights = make_class_weights(y_train.tolist()).to(device)
    surname_criterion = nn.CrossEntropyLoss(weight=class_weights)

    fixed_embedding = nn.Embedding.from_pretrained(torch.eye(char_vocab.vocab_len), freeze=True, padding_idx=0)
    surname_model_fixed = SurnameConvClassifier(fixed_embedding, char_vocab.vocab_len, num_classes)
    optimizer_fixed = optim.Adam(surname_model_fixed.parameters(), lr=1e-3, weight_decay=1e-4)
    history_fixed = train_classifier(surname_model_fixed, train_loader, test_loader, surname_criterion, optimizer_fixed, epochs=12, device=device, patience=3, min_delta=1e-3)
    plot_history(history_fixed, 'Surname ConvNet (fixed one-hot embedding)')
    fixed_loss, fixed_acc = evaluate_classifier(surname_model_fixed, test_loader, surname_criterion, device)
    print('Fixed embedding accuracy:', fixed_acc)
    clear_memory(surname_model_fixed, optimizer_fixed, history_fixed)

    trainable_embedding = nn.Embedding(char_vocab.vocab_len, 32, padding_idx=0)
    surname_model_trainable = SurnameConvClassifier(trainable_embedding, 32, num_classes)
    optimizer_trainable = optim.Adam(surname_model_trainable.parameters(), lr=1e-3, weight_decay=1e-4)
    history_trainable = train_classifier(surname_model_trainable, train_loader, test_loader, surname_criterion, optimizer_trainable, epochs=15, device=device, patience=4, min_delta=1e-3)
    plot_history(history_trainable, 'Surname ConvNet (trainable embedding)')
    trainable_loss, trainable_acc = evaluate_classifier(surname_model_trainable, test_loader, surname_criterion, device)
    print('Trainable embedding accuracy:', trainable_acc)

    def decode_nationality(idx):
        return label_encoder.inverse_transform([idx])[0]

    def predict_surname(model, surname, dataset, top_k=3):
        model.eval()
        with torch.no_grad():
            vector = dataset.vectorize(surname).unsqueeze(0).to(device)
            probs = torch.softmax(model(vector), dim=1).squeeze(0)
            top_probs, top_idx = torch.topk(probs, k=top_k)
        return [(decode_nationality(int(idx)), float(prob)) for prob, idx in zip(top_probs.cpu(), top_idx.cpu())]

    for surname in ['Ivanov', 'Petrov', 'Smith', 'Kim', 'Garcia']:
        print(surname, '->', predict_surname(surname_model_trainable, surname, train_dataset))

    torch.save(surname_model_trainable.state_dict(), MODELS_DIR / 'surname_conv_classifier.pth')
    clear_memory(surname_model_trainable, optimizer_trainable, history_trainable, surname_criterion)


## 3. Классификация обзоров на фильмы (ConvNet)

Датасет: https://disk.yandex.ru/d/tdinpb0nN_Dsrg

2.1 Создайте набор данных на основе файлов polarity/positive_reviews.csv (положительные отзывы) и polarity/negative_reviews.csv (отрицательные отзывы). Разбейте на обучающую и тестовую выборку.
  * токен = __слово__
  * данные для обучения в датасете представляются в виде последовательности индексов токенов
  * словарь создается на основе _только_ обучающей выборки. Для корректной обработки ситуаций, когда в тестовой выборке встретится токен, который не хранится в словаре, добавьте в словарь специальный токен `<UNK>`
  * добавьте предобработку текста

2.2. Обучите классификатор.
  
  * Для преобразования последовательности индексов в последовательность векторов используйте `nn.Embedding` 
    - подберите адекватную размерность вектора эмбеддинга: 
    - модуль `nn.Embedding` обучается

  * Используйте одномерные свертки и пулинг (`nn.Conv1d`, `nn.MaxPool1d`)
    - обратите внимание, что `nn.Conv1d` ожидает на вход трехмерный тензор размерности `(batch, embedding_dim, seq_len)`


2.7 Измерить точность на тестовой выборке. Проверить работоспособность модели: придумать небольшой отзыв, прогнать его через модель и вывести номер предсказанного класса (сделать это для явно позитивного и явно негативного отзыва)
* Целевое значение accuracy на валидации - 70+%

In [ ]:
positive_candidates = [
    DATA_DIR / 'polarity' / 'positive_reviews.csv',
    DATA_DIR / 'positive_reviews.csv',
]
negative_candidates = [
    DATA_DIR / 'polarity' / 'negative_reviews.csv',
    DATA_DIR / 'negative_reviews.csv',
]

positive_path = next((path for path in positive_candidates if path.exists()), None)
negative_path = next((path for path in negative_candidates if path.exists()), None)

if positive_path is None or negative_path is None:
    print('Не найдены файлы polarity. Ожидаемые пути:', positive_candidates, negative_candidates)
else:
    positive_df = pd.read_csv(positive_path, header=None, names=['text'])
    negative_df = pd.read_csv(negative_path, header=None, names=['text'])
    positive_df['label'] = 1
    negative_df['label'] = 0

    reviews_df = pd.concat([positive_df, negative_df], ignore_index=True)
    reviews_df['text'] = reviews_df['text'].astype(str)
    reviews_df['processed_tokens'] = reviews_df['text'].apply(preprocess_text)

    X_train, X_test, y_train, y_test = train_test_split(
        reviews_df['processed_tokens'],
        reviews_df['label'],
        test_size=0.2,
        random_state=42,
        stratify=reviews_df['label'],
    )

    train_tokens = [token for tokens in X_train for token in tokens]
    review_vocab = Vocab(train_tokens, pad_token='<PAD>', unk_token='<UNK>', min_freq=5)
    review_vocab.max_seq_len = min(300, int(X_train.map(len).quantile(0.95)))
    review_vocab.max_seq_len = max(review_vocab.max_seq_len, 20)

    class ReviewDataset(Dataset):
        def __init__(self, X, y, vocab: Vocab):
            self.X = list(X)
            self.y = list(y)
            self.vocab = vocab

        def vectorize(self, tokens):
            pad_idx = self.vocab.token_to_idx[self.vocab.pad_token]
            unk_idx = self.vocab.token_to_idx[self.vocab.unk_token]
            indices = [self.vocab.token_to_idx.get(token, unk_idx) for token in tokens[:self.vocab.max_seq_len]]
            if len(indices) < self.vocab.max_seq_len:
                indices += [pad_idx] * (self.vocab.max_seq_len - len(indices))
            return torch.tensor(indices, dtype=torch.long)

        def __len__(self):
            return len(self.X)

        def __getitem__(self, idx):
            return self.vectorize(self.X[idx]), torch.tensor(self.y[idx], dtype=torch.long)

    review_train_dataset = ReviewDataset(X_train.tolist(), y_train.tolist(), review_vocab)
    review_test_dataset = ReviewDataset(X_test.tolist(), y_test.tolist(), review_vocab)

    review_train_loader = DataLoader(review_train_dataset, batch_size=256, shuffle=True)
    review_test_loader = DataLoader(review_test_dataset, batch_size=512, shuffle=False)

    class ReviewConvClassifier(nn.Module):
        def __init__(self, vocab_size, emb_dim, num_classes, pad_idx):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
            self.conv3 = nn.Conv1d(emb_dim, 128, kernel_size=3, padding=1)
            self.conv5 = nn.Conv1d(emb_dim, 128, kernel_size=5, padding=2)
            self.conv7 = nn.Conv1d(emb_dim, 128, kernel_size=7, padding=3)
            self.pool = nn.AdaptiveMaxPool1d(1)
            self.dropout = nn.Dropout(0.5)
            self.fc = nn.Linear(128 * 3, num_classes)

        def forward(self, x):
            x = self.embedding(x)
            x = x.transpose(1, 2)
            x3 = self.pool(torch.relu(self.conv3(x))).squeeze(-1)
            x5 = self.pool(torch.relu(self.conv5(x))).squeeze(-1)
            x7 = self.pool(torch.relu(self.conv7(x))).squeeze(-1)
            x = torch.cat([x3, x5, x7], dim=1)
            x = self.dropout(x)
            return self.fc(x)

    review_model = ReviewConvClassifier(review_vocab.vocab_len, 64, 2, review_vocab.token_to_idx['<PAD>'])
    review_criterion = nn.CrossEntropyLoss()
    review_optimizer = optim.Adam(review_model.parameters(), lr=7e-4, weight_decay=1e-4)
    history_reviews = train_classifier(review_model, review_train_loader, review_test_loader, review_criterion, review_optimizer, epochs=10, device=device, patience=2, min_delta=1e-3)
    plot_history(history_reviews, 'Review ConvNet')
    review_loss, review_acc = evaluate_classifier(review_model, review_test_loader, review_criterion, device)
    print('Review ConvNet accuracy:', review_acc)

    def decode_review_label(idx):
        return 'positive' if idx == 1 else 'negative'

    show_text_predictions(review_model, X_test.tolist(), y_test.tolist(), review_test_dataset.vectorize, decode_review_label, device, n=5, title='Movie review test examples')

    def predict_review_text(model, text):
        model.eval()
        with torch.no_grad():
            tokens = preprocess_text(text)
            vector = review_test_dataset.vectorize(tokens).unsqueeze(0).to(device)
            pred = int(torch.argmax(model(vector), dim=1).item())
        return pred

    positive_review = 'Amazing movie, brilliant acting and wonderful atmosphere. I loved every minute of it.'
    negative_review = 'Terrible movie, boring plot and awful acting. I regret watching it.'
    print('positive_review ->', predict_review_text(review_model, positive_review))
    print('negative_review ->', predict_review_text(review_model, negative_review))

    torch.save(review_model.state_dict(), MODELS_DIR / 'review_conv_classifier.pth')
    clear_memory(review_model, history_reviews, review_optimizer, review_criterion)
